In [1]:
!pip install -U transformers accelerate datasets peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 8.8 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 41.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(hf_token)

In [3]:
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import random

In [4]:
MODEL_ID = "google/medgemma-1.5-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False

config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

In [5]:
chatdoctor = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
wikidoc = load_dataset("medalpaca/medical_meadow_wikidoc_patient_information", split="train")

README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc_patient_info.json:   0%|          | 0.00/3.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5942 [00:00<?, ? examples/s]

In [6]:
KEYWORDS = [
    "pregnancy","antenatal","postpartum","newborn","neonate","breastfeed",
    "jaundice","preeclampsia","gestational diabetes","anaemia","low birth weight",
    "cord","lactation","miscarriage","ectopic","folic acid","iron"
]

def keyword_filter(example):
    text = (example["input"] + " " + example["output"]).lower()
    return any(k in text for k in KEYWORDS)

chatdoctor = chatdoctor.filter(keyword_filter)
wikidoc = wikidoc.filter(keyword_filter)

chatdoctor = chatdoctor.select(range(min(5000, len(chatdoctor))))
wikidoc = wikidoc.select(range(min(1500, len(wikidoc))))

dataset = concatenate_datasets([chatdoctor, wikidoc]).shuffle(seed=42)

Filter:   0%|          | 0/112165 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5942 [00:00<?, ? examples/s]

In [7]:
SYSTEM_PROMPT = """You are a knowledgeable clinical assistant specialising in maternal health,
antenatal care, postpartum care, and newborn health. Provide accurate,
evidence-based answers. Recommend medical attention if serious symptoms appear.
Never speculate beyond your knowledge."""

In [8]:
def format_chat(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]},
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    
    return {"text": text}

dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=0.05, seed=42)

Map:   0%|          | 0/5602 [00:00<?, ? examples/s]

In [9]:
print(dataset["train"][0]["text"][:500])

<bos><start_of_turn>user
You are a knowledgeable clinical assistant specialising in maternal health,
antenatal care, postpartum care, and newborn health. Provide accurate,
evidence-based answers. Recommend medical attention if serious symptoms appear.
Never speculate beyond your knowledge.

What are the symptoms of Gestational diabetes?<end_of_turn>
<start_of_turn>model
Usually there are no symptoms, or the symptoms are mild and not life threatening to the pregnant woman. Often, the blood sugar 


In [10]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

In [11]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )

    # REQUIRED for Gemma 3 training
    tokens["token_type_ids"] = [0] * len(tokens["input_ids"])

    # Causal LM labels
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

dataset = dataset.map(
    tokenize_function,
    remove_columns=["text"]
)

dataset.set_format("torch")

Map:   0%|          | 0/5321 [00:00<?, ? examples/s]

Map:   0%|          | 0/281 [00:00<?, ? examples/s]

In [12]:
print(dataset["train"][0].keys())
dataset["train"] = dataset["train"].select(range(2500))

dict_keys(['input_ids', 'attention_mask', 'token_type_ids', 'labels'])


In [13]:
from peft import prepare_model_for_kbit_training, get_peft_model

model.gradient_checkpointing_enable()

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

trainable params: 38,497,792 || all params: 4,338,577,264 || trainable%: 0.8873


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="sakhi-medgemma-4b-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=50,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args,
)

In [16]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,2.610216
100,2.104061
150,2.031777
200,2.027452
250,2.013540
300,2.044435


TrainOutput(global_step=313, training_loss=2.130927661737314, metrics={'train_runtime': 15247.2011, 'train_samples_per_second': 0.164, 'train_steps_per_second': 0.021, 'total_flos': 2.812881555456e+16, 'train_loss': 2.130927661737314, 'epoch': 1.0})

In [17]:
trainer.model.save_pretrained("sakhi-medgemma-lora")
tokenizer.save_pretrained("sakhi-medgemma-lora")

('sakhi-medgemma-lora/tokenizer_config.json',
 'sakhi-medgemma-lora/chat_template.jinja',
 'sakhi-medgemma-lora/tokenizer.json')

In [18]:
HF_USERNAME = "docvm"
REPO_ID = f"{HF_USERNAME}/sakhi-medgemma-1.5-4b-maternal"

In [19]:
model_card = f"""
---
base_model: google/medgemma-1.5-4b-it
tags:
- medgemma
- lora
- maternal-health
- neonatal-care
- india-healthcare
- qlora
license: gemma
---

# Sakhi – MedGemma 1.5 4B Maternal Health LoRA

This repository contains a LoRA adapter fine-tuned on maternal, antenatal, postpartum, and newborn health data for ASHA worker clinical assistance.

Base model: google/medgemma-1.5-4b-it

---

## Training Method

- Base model loaded in 4-bit NF4 quantization (QLoRA)
- LoRA rank: 16
- LoRA alpha: 16
- Optimizer: paged_adamw_8bit
- Learning rate: 2e-4
- Epochs: 1
- Max sequence length: 1024
- Hardware: Kaggle 2×T4 GPUs

Datasets:
- lavita/ChatDoctor-HealthCareMagic-100k (filtered maternal/newborn)
- medalpaca/medical_meadow_wikidoc_patient_information (OB/GYN filtered)

---

## Intended Use

This adapter is designed to improve maternal and neonatal health reasoning for:

- Antenatal care interpretation
- High-risk pregnancy identification
- Neonatal danger sign detection
- Breastfeeding and postpartum support
- India-specific maternal health contexts (IFA, anaemia, PHC referral)

It is intended for use inside Sakhi — an AI clinical assistant for ASHA workers.

---

## Prohibited Use

This model must NOT be used for:

- Autonomous medical diagnosis
- Replacing licensed medical professionals
- Emergency decision-making without human oversight
- Direct-to-patient unsupervised deployment

---

## Safety Limitations

- Model may hallucinate.
- Does not replace trained clinicians.
- Should be used only as decision-support.
- Must recommend PHC referral when uncertainty exists.

---

## System Prompts

### CHECKUP Mode

Respond strictly in JSON format:

{{
  "risk_level": "green" | "yellow" | "red",
  "risk_reason": "...",
  "what_sakhi_noticed": ["...", "..."],
  "what_to_tell_patient": "...",
  "what_to_do_next": "...",
  "follow_up_date": "YYYY-MM-DD or null"
}}

Red flag example:
BP > 140/90 → risk_level: red

---

### NEWBORN Mode

Age-specific rules:
- Jaundice after Day 14 → red
- Weight loss after Day 7 → yellow/red
- Cord redness or smell → red
- Breathing > 60/min → red

Same JSON schema as CHECKUP mode.

---

### CHAT Mode

Conversational.
Always address the ASHA worker directly.
Use simple, clear English.
Recommend PHC referral when unsure.

---

## License

Gemma Terms of Use apply.
"""

In [20]:
with open("sakhi-medgemma-lora/README.md", "w") as f:
    f.write(model_card)

In [21]:
from huggingface_hub import create_repo

create_repo(REPO_ID, private=True, exist_ok=True)

trainer.model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/docvm/sakhi-medgemma-1.5-4b-maternal/commit/12374cb2eae80975c6f3e33b685bca177a2b4497', commit_message='Upload tokenizer', commit_description='', oid='12374cb2eae80975c6f3e33b685bca177a2b4497', pr_url=None, repo_url=RepoUrl('https://huggingface.co/docvm/sakhi-medgemma-1.5-4b-maternal', endpoint='https://huggingface.co', repo_type='model', repo_id='docvm/sakhi-medgemma-1.5-4b-maternal'), pr_revision=None, pr_num=None)